In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.optimize import curve_fit
import matplotlib
matplotlib.use('Agg')  # suppress plots while we rebuild the data
import matplotlib.pyplot as plt

# ── Replicate Tom's pipeline up to curve fitting only ──

path = Path('cashflows_with_sq')
projects = {}
for file in sorted(path.glob('*.csv'), key=lambda p: p.name):
    current_df = pd.read_csv(file)
    projects[file.stem] = current_df

# Clean + align to master timeline
for name, df in projects.items():
    df_cleaned = df.iloc[:, 1:].copy()
    if len(df_cleaned) > 0:
        df_cleaned.iat[-1, 0] = "Total"
        start_col_idx = 6
        if df_cleaned.shape[1] > start_col_idx:
            last_row = df_cleaned.iloc[[-1], start_col_idx:]
            cleaned_last_row = (
                last_row.astype("string")
                .apply(lambda s: s.str.split(r"\r\n|\n|\r", regex=True).str[0])
            )
            df_cleaned.iloc[-1, start_col_idx:] = cleaned_last_row.iloc[0].to_numpy()
    df_cleaned = df_cleaned.reset_index(drop=True)
    projects[name] = df_cleaned

timeline_dates = pd.date_range(start='2011-01-01', end='2027-06-01', freq='MS')
master_timeline = timeline_dates.strftime('%b %Y').tolist()

for name, df in projects.items():
    static_columns = list(df.columns[:7])
    df_unified = df.reindex(columns=static_columns + master_timeline, fill_value=0)
    projects[name] = df_unified

# Keep totals only
for name, df in projects.items():
    total_only = df[df.iloc[:, 0].astype(str).eq("Total")].copy()
    total_only = total_only.drop(columns=["Line Item", "Description"])
    projects[name] = total_only.reset_index(drop=True)

# Add project code
for name, df in projects.items():
    if df.empty: continue
    if "Project Code" in df.columns:
        df = df.drop(columns=["Project Code"])
    df.insert(0, "Project Code", name)
    projects[name] = df

# Combine
import os
non_empty = [df for df in projects.values() if not df.empty]
all_projects_df = pd.concat(non_empty, ignore_index=True)

# Merge gross sq footage
gross_sq_path = os.path.join(path, "gross_sq.csv") if os.path.exists(
    os.path.join(path, "gross_sq.csv")) else "gross_sq.csv"
gross_sq = pd.read_csv(gross_sq_path)
all_projects_df = all_projects_df.merge(
    gross_sq[["Project Code", "Gross Sq Footage"]], on="Project Code", how="left")
cols = list(all_projects_df.columns)
cols.insert(1, cols.pop(cols.index("Gross Sq Footage")))
all_projects_df = all_projects_df[cols]
all_projects_df = all_projects_df.drop(
    columns=["Actuals To Date", "Actuals + Projections"], errors="ignore")

# Cumulative sum
timeline_cols = all_projects_df.columns[5:]
all_projects_df[timeline_cols] = (
    all_projects_df[timeline_cols]
    .astype(str)
    .replace({',': '', r'\(': '-', r'\)': ''}, regex=True)
    .astype(float)
)
all_projects_df[timeline_cols] = all_projects_df[timeline_cols].cumsum(axis=1)
all_projects_df = all_projects_df[
    all_projects_df["Project Code"].astype(str) != "5149"].reset_index(drop=True)

# ── Fit S-curves (no plotting) ──
def logistic_curve(t, L, k, t0):
    return L / (1 + np.exp(-k * (t - t0)))

all_projects_df_2 = all_projects_df.copy()
timeline_cols_list = timeline_dates.strftime('%b %Y').tolist()

L_params, k_params, t0_params, durations, r2_values = [], [], [], [], []
R2_THRESHOLD = 0.75

for index, row in all_projects_df_2.iterrows():
    y_data_full = row[timeline_cols_list].values.astype(float)
    start_idx = np.argmax(y_data_full > 0)
    monthly_deltas = np.diff(y_data_full, prepend=0)
    nonzero_months = np.where(monthly_deltas > 0)[0]

    if len(nonzero_months) == 0:
        durations.append(np.nan); L_params.append(np.nan)
        k_params.append(np.nan); t0_params.append(np.nan)
        r2_values.append(np.nan); continue

    stop_idx = nonzero_months[-1]
    num_months = stop_idx - start_idx + 1
    durations.append(num_months)

    if num_months < 3:
        L_params.append(np.nan); k_params.append(np.nan)
        t0_params.append(np.nan); r2_values.append(np.nan); continue

    y_active = y_data_full[start_idx: stop_idx + 1]
    x_active = np.arange(num_months)

    # Clamp L_init so initial guess is always strictly inside bounds
    L_init = max(float(y_active[-1]), 1.0)
    max_budget_limit = L_init * 1.5 + 1.0  # +1 guarantees upper > lower even if L_init tiny
    
    initial_guess = [L_init, 0.1, num_months / 2]
    param_bounds = ([0, 0.001, 0], [max_budget_limit, 5.0, num_months])

    try:
        popt, _ = curve_fit(logistic_curve, x_active, y_active,
                            p0=initial_guess, bounds=param_bounds, maxfev=5000)
        y_pred_fit = logistic_curve(x_active, *popt)
        ss_res = np.sum((y_active - y_pred_fit) ** 2)
        ss_tot = np.sum((y_active - np.mean(y_active)) ** 2)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
        r2_values.append(r_squared)
        if r_squared < R2_THRESHOLD:
            L_params.append(np.nan); k_params.append(np.nan); t0_params.append(np.nan)
        else:
            L_params.append(popt[0]); k_params.append(popt[1]); t0_params.append(popt[2])
    except (RuntimeError, ValueError):
        L_params.append(np.nan); k_params.append(np.nan)
        t0_params.append(np.nan); r2_values.append(np.nan)

all_projects_df_2['Duration_Months'] = durations
all_projects_df_2['S_Curve_L']       = L_params
all_projects_df_2['S_Curve_k']       = k_params
all_projects_df_2['S_Curve_t0']      = t0_params
all_projects_df_2['Fit_R2']          = r2_values

# Restore normal plot rendering
matplotlib.use('module://matplotlib_inline.backend_inline')

print(f"Pipeline complete. Projects with fitted curves: "
      f"{all_projects_df_2['S_Curve_k'].notna().sum()} / {len(all_projects_df_2)}")
print(all_projects_df_2[['Project Code','S_Curve_k','S_Curve_t0','Duration_Months','Fit_R2']].dropna().head())

Pipeline complete. Projects with fitted curves: 211 / 237
  Project Code  S_Curve_k  S_Curve_t0  Duration_Months    Fit_R2
0         5004   0.691515   13.338872             35.0  0.997183
1         5018   0.113672   15.090819             37.0  0.852584
2         5046   0.135723   16.353815            119.0  0.963829
3         5058   0.197593   38.709869             67.0  0.997022
4         5088   0.200762   47.672112             67.0  0.998320


/var/folders/ng/dp7x4ps91zd04cxs5zk8k5fw0000gn/T/ipykernel_85983/3534516886.py:146: MatplotlibDeprecationWarning: Auto-close()ing of figures upon backend switching is deprecated since 3.8 and will be removed in 3.10.  To suppress this warning, explicitly call plt.close('all') first.
  matplotlib.use('module://matplotlib_inline.backend_inline')


In [ ]:
# eda_tier_analysis.ipynb
# Run AFTER training_notebook_totals.ipynb and training_notebook_line6.ipynb
# Assumes all_projects_df_2 (with fitted params) is available in memory,
# OR re-run the fitting section to regenerate it.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import json

# ─────────────────────────────────────────────
# 1. ATTACH SIZE TIERS
# ─────────────────────────────────────────────
def assign_tier(budget):
    if pd.isna(budget):   return "Unknown"
    if budget < 250_000:  return "Small"
    if budget < 5_000_000: return "Medium"
    if budget < 50_000_000: return "Large"
    return "Mega"

# Use Projected Commitments as the budget proxy (same as Tom's L assignment)
timeline_dates_drop = pd.date_range(start='2011-01-01', end='2027-06-01', freq='MS')
timeline_cols_drop = timeline_dates_drop.strftime('%b %Y').tolist()

df = all_projects_df_2.drop(columns=timeline_cols_drop, errors='ignore').copy()  # the post-training df with S_Curve params
df["Budget"] = (
    df["Projected Commitments"]
    .astype(str)
    .str.replace(r"[\$,]", "", regex=True)
    .astype(float)
)
df["Size_Tier"] = df["Budget"].apply(assign_tier)

# Drop rows where the curve fit failed (NaN params) — these aren't useful for analysis
df_valid = df.dropna(subset=["S_Curve_k", "S_Curve_t0", "Duration_Months"]).copy()

print(f"Valid fitted projects: {len(df_valid)}")
print(df_valid["Size_Tier"].value_counts())


# ─────────────────────────────────────────────
# 2. GROWTH RATE vs PROJECT SIZE
# Chris asked: "how does project size change what the data says about cash spend"
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

tier_colors = {"Small": "#3b82f6", "Medium": "#f59e0b", "Large": "#ef4444", "Mega": "#8b5cf6"}

for tier, grp in df_valid.groupby("Size_Tier"):
    ax.scatter(grp["Budget"], grp["S_Curve_k"],
               label=tier, color=tier_colors.get(tier, "gray"),
               alpha=0.7, s=60, zorder=3)

ax.set_xscale("log")
ax.set_xlabel("Project Budget (log scale)", fontsize=12)
ax.set_ylabel("S-Curve Growth Rate (k)", fontsize=12)
ax.set_title("Growth Rate vs. Project Size", fontsize=14, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"${x/1e6:.0f}M" if x >= 1e6 else f"${x/1e3:.0f}K"
))
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend(title="Tier")
plt.tight_layout()
plt.savefig("outputs/growth_rate_vs_size.png", dpi=150)
plt.show()


# ─────────────────────────────────────────────
# 3. NORMALIZED MIDPOINT vs PROJECT SIZE
# t0/Duration = where in the project lifecycle does spending peak
# ─────────────────────────────────────────────
df_valid["t0_normalized"] = df_valid["S_Curve_t0"] / df_valid["Duration_Months"]

fig, ax = plt.subplots(figsize=(9, 5))
for tier, grp in df_valid.groupby("Size_Tier"):
    ax.scatter(grp["Budget"], grp["t0_normalized"],
               label=tier, color=tier_colors.get(tier, "gray"),
               alpha=0.7, s=60, zorder=3)

ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="Midpoint = 50%")
ax.set_xscale("log")
ax.set_xlabel("Project Budget (log scale)", fontsize=12)
ax.set_ylabel("Spending Peak (fraction of project lifetime)", fontsize=12)
ax.set_title("When Does Spending Peak? Midpoint vs. Project Size", fontsize=14, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"${x/1e6:.0f}M" if x >= 1e6 else f"${x/1e3:.0f}K"
))
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend(title="Tier")
plt.tight_layout()
plt.savefig("outputs/midpoint_vs_size.png", dpi=150)
plt.show()


# ─────────────────────────────────────────────
# 4. DISTRIBUTION OF k AND t0 BY TIER (boxplots)
# This is the "variation" visualization Chris asked for
# ─────────────────────────────────────────────
tier_order = ["Small", "Medium", "Large", "Mega"]
tiers_present = [t for t in tier_order if t in df_valid["Size_Tier"].values]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# k by tier
data_k = [df_valid[df_valid["Size_Tier"] == t]["S_Curve_k"].dropna().values for t in tiers_present]
bp1 = axes[0].boxplot(data_k, labels=tiers_present, patch_artist=True)
for patch, tier in zip(bp1["boxes"], tiers_present):
    patch.set_facecolor(tier_colors.get(tier, "gray"))
    patch.set_alpha(0.7)
axes[0].set_title("Growth Rate (k) by Project Tier", fontweight="bold")
axes[0].set_ylabel("S-Curve k parameter")
axes[0].grid(True, axis="y", linestyle="--", alpha=0.4)

# t0 normalized by tier
data_t0 = [df_valid[df_valid["Size_Tier"] == t]["t0_normalized"].dropna().values for t in tiers_present]
bp2 = axes[1].boxplot(data_t0, labels=tiers_present, patch_artist=True)
for patch, tier in zip(bp2["boxes"], tiers_present):
    patch.set_facecolor(tier_colors.get(tier, "gray"))
    patch.set_alpha(0.7)
axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=1)
axes[1].set_title("Spending Peak (t0/Duration) by Project Tier", fontweight="bold")
axes[1].set_ylabel("Fraction of project lifetime")
axes[1].grid(True, axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("outputs/tier_distributions.png", dpi=150)
plt.show()


# ─────────────────────────────────────────────
# 5. PERCENTILE ENVELOPE CURVES PER TIER
# The confidence band Chris asked for — run this per tier
# ─────────────────────────────────────────────
def logistic_curve(t, L, k, t0):
    return L / (1 + np.exp(-k * (t - t0)))

def plot_tier_envelope(tier_name, df_tier, n_points=100):
    """
    For all valid projects in a tier, normalize each curve to [0,1] time axis,
    compute 10th/50th/90th percentile envelopes.
    """
    if len(df_tier) < 3:
        print(f"  Skipping {tier_name}: too few projects ({len(df_tier)})")
        return

    normalized_curves = []
    t_norm = np.linspace(0, 1, n_points)

    for _, row in df_tier.iterrows():
        dur = int(row["Duration_Months"])
        L   = row["Budget"]  # use actual budget as ceiling
        k   = row["S_Curve_k"]
        t0  = row["S_Curve_t0"]
        t_raw = np.linspace(0, dur - 1, n_points)
        curve = logistic_curve(t_raw, L, k, t0)
        # Normalize to 0-1 scale (% of total budget)
        curve_norm = curve / L if L > 0 else curve
        normalized_curves.append(curve_norm)

    curves_arr = np.array(normalized_curves)
    p10 = np.percentile(curves_arr, 10, axis=0)
    p50 = np.percentile(curves_arr, 50, axis=0)
    p90 = np.percentile(curves_arr, 90, axis=0)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.fill_between(t_norm * 100, p10 * 100, p90 * 100,
                    alpha=0.2, color=tier_colors.get(tier_name, "blue"),
                    label="10th–90th percentile")
    ax.plot(t_norm * 100, p50 * 100,
            color=tier_colors.get(tier_name, "blue"), linewidth=2.5,
            label="Median (50th percentile)")
    ax.set_xlabel("Project Timeline (%)", fontsize=12)
    ax.set_ylabel("Cumulative Spend (% of Budget)", fontsize=12)
    ax.set_title(f"{tier_name} Projects: S-Curve Envelope\n(n={len(df_tier)} projects)",
                 fontsize=14, fontweight="bold")
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 105)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"outputs/envelope_{tier_name.lower()}.png", dpi=150)
    plt.show()

import os
os.makedirs("outputs", exist_ok=True)

for tier in tiers_present:
    plot_tier_envelope(tier, df_valid[df_valid["Size_Tier"] == tier])


# ─────────────────────────────────────────────
# 6. EXPORT SUMMARY STATS JSON FOR WEBSITE
# Chris asked: "put those summary stats of model training onto the website"
# ─────────────────────────────────────────────
summary_stats = {}

for tier in tiers_present:
    grp = df_valid[df_valid["Size_Tier"] == tier]
    summary_stats[tier] = {
        "n_projects":          int(len(grp)),
        "median_budget_usd":   round(grp["Budget"].median(), 0),
        "min_budget_usd":      round(grp["Budget"].min(), 0),
        "max_budget_usd":      round(grp["Budget"].max(), 0),
        "median_duration_months": round(grp["Duration_Months"].median(), 1),
        "min_duration_months":    round(grp["Duration_Months"].min(), 1),
        "max_duration_months":    round(grp["Duration_Months"].max(), 1),
        "median_k":            round(float(grp["S_Curve_k"].median()), 4),
        "median_t0_pct":       round(float(grp["t0_normalized"].median()) * 100, 1),
        "model_median_r2":     round(float(grp["Fit_R2"].median()), 3),
    }

# Overall
summary_stats["_overall"] = {
    "n_projects":        int(len(df_valid)),
    "data_range":        "2011–2025",
    "total_portfolio_usd": round(df_valid["Budget"].sum(), 0),
}

with open("outputs/training_summary_stats.json", "w") as f:
    json.dump(summary_stats, f, indent=2)

print(json.dumps(summary_stats, indent=2))

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)